**MSB1015 - Maastricht University**

The point of this Jupyter notebook is to make a machine learning prediction model of the 2025 season of F1. Using data from previous years, I intend to: Predict the 2025 world champion and the upcoming races, see which drivers further back in the grid are making notable strides of improvement, and maybe predict who will be a future world champion after the new regulation changes in 2026. In order to complete this task, the following pipeline will be applied.

Data merging
--> I wanna create a table where, for each round in a season.

First step is to create a table that normalizes points received each season. Since each season has a different amount of races and also received less points per race, these races would be deemed worth less artificially.

However, it is definitely easier to first create a table that for each driver just says their driverID, raceID, constructorID, etc.

Main table:
driverId, driverRef, resultId, raceId, circuitId, year, constructorId, grid, positionOrder, points

In [ ]:
# making a table for each drivers race results, combine results.csv with drivers.csv first
# then combine

# load the necessary libraries
import pandas as pd
import numpy as np

# load the datasets results.csv and drivers.csv and race.csv
results_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\results.csv")
drivers_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\drivers.csv")
races_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\races.csv")

drivers_df = drivers_df[["driverId", "driverRef"]]
races_df = races_df[["raceId", "year", "round", "circuitId", "name"]]

# merge the datasets on driverId
combined_df = pd.merge(results_df, drivers_df, on="driverId")
combined_df = combined_df[["driverId", "driverRef", "resultId", "raceId", "constructorId", "grid", "positionOrder", "points"]]

# rename grid to starting_position and positionOrder to finishpos
combined_df = combined_df.rename(columns={"grid": "startPos"})
combined_df = combined_df.rename(columns={"positionOrder": "finishPos"})

combined_df = pd.merge(combined_df, races_df, on="raceId")

# add a normalizer of the points to the maximum of each race so that each race has a maximum of 1 point
combined_df["nPoints"] = combined_df.groupby("raceId")["points"].transform(lambda x: x / x.max())




# save the combined dataset to a new csv file
combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results.csv", index=False)

del combined_df
del results_df
del drivers_df
del races_df

Now that we got a good starting dataset, lets start adding some race specific and in general messier data

Most importantly, I need to find out a way to add a column that marks someone the world champion, so the AI can learn off that. Just adding one label at the last race doesnt work obviously, and I want it to look at the performances from early races to see how the world champion of that year performs. 
--> Solution, add a label to each row if hes the world champion or not, so the AI can see for each race how he performed for each year.

First we need a table that shows for each year who was world champion

In [33]:
# maka a table that for each year says who was the world champion based on combined_df, the sum of points per year
combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results.csv")

world_champions = combined_df.groupby(["year", "driverId", "driverRef"])["points"].sum().reset_index()
world_champions = world_champions.loc[world_champions.groupby("year")["points"].idxmax()]
# keep only the columns year, driverId, driverRef
world_champions = world_champions[["year", "driverId", "driverRef"]]

# save to csv
world_champions.to_csv("d:\\Maastricht University\\MSB1015\\interval\\world_champions.csv", index=False)

del world_champions
del combined_df


Now make a column for each row in the combined_df that shows if the driverId is a world champion or not

In [ ]:
# add a column that indicates if the driver is a world champion or not
combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results.csv")
world_champions = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\world_champions.csv")

combined_df = pd.merge(combined_df, world_champions[["year", "driverId"]].assign(worldChamp=1), on=["year", "driverId"], how="left")
combined_df["worldChamp"] = combined_df["worldChamp"].fillna(0)

# put worldChamp first in the columns
cols = combined_df.columns.tolist()
cols = cols[-1:] + cols[:-1]
combined_df = combined_df[cols]

# save the combined dataset to a new csv file
combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_champions.csv", index=False)

del combined_df
del world_champions
del cols

Now that we have added the label that we will eventually train on, lets add more data that might be interesting. I will keep the world_champion tag near the end of the dataset.

Things to add:
* qualifying pace from qualifying
* pitstops (issue, each pitstop is noted individually, maybe just total # of stops)
* avg laptime from lap_time so we can stratify for car performance as well
* driver standings from driver_standings
* sprint race results

In [ ]:
# now that we have a good starting dataset, we can start adding more features to it
combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_champions.csv")

# qualifying pace from qualifying.csv
qualifying_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\qualifying.csv")
qualifying_df = qualifying_df[["raceId", "driverId", "q1", "q2", "q3"]]

# qualifying times are in the format "1:23.456", we need to convert them to seconds. Some values are NaN or '\N' if the driver was knocked out in Q1 or Q2
def convert_to_seconds(time_str):
    if pd.isna(time_str) or time_str == r'\N':
        return np.nan
    if ':' in time_str:
        minutes, seconds = time_str.split(':')
        return int(minutes) * 60 + float(seconds)
    else:
        return float(time_str)

# apply the conversion function to each qualifying time column
qualifying_df["q1"] = qualifying_df["q1"].apply(convert_to_seconds)
qualifying_df["q2"] = qualifying_df["q2"].apply(convert_to_seconds)
qualifying_df["q3"] = qualifying_df["q3"].apply(convert_to_seconds)

# save qualifying_df to csv
qualifying_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\qualifying_times.csv", index=False)

# add qualifying times to combined_df
combined_df = pd.merge(combined_df, qualifying_df, on=["raceId", "driverId"], how="left")

# save the combined dataset to a new csv file
combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_qualifying.csv", index=False)

del combined_df
del qualifying_df

In [ ]:
# load pit stop data
pit_stops_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\pit_stops.csv")
pit_stops_df = pit_stops_df[["raceId", "driverId", "stop", "duration"]]
pit_stops_df['stop'] = pit_stops_df['stop'].astype(int)
pit_stops_df['duration'] = pd.to_numeric(pit_stops_df['duration'], errors='coerce')

combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_qualifying.csv")

# number of pitstops and average pitstop duration with new column names
pit_features = pit_stops_df.groupby(['raceId', 'driverId']).agg(
    nPits=('stop', 'count'),
    pitDur=('duration', 'mean')
).reset_index()

# remove any old pitstop columns before merging
pitstop_cols = [col for col in combined_df.columns if col in ['nPits', 'pitDur']]
combined_df = combined_df.drop(columns=pitstop_cols, errors='ignore')

# merge features
combined_df = pd.merge(combined_df, pit_features, on=["raceId", "driverId"], how="left")

# save the combined dataset to a new csv file
combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_pitstops.csv", index=False)

del combined_df
del pit_stops_df
del pit_features
del pitstop_cols

In [37]:
# average speed from lap_times.csv. lap_times.csv shows the raceId, driverId, lap, position, time, and milliseconds. Calculate the fastest lap time for each driver in each race 
combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_pitstops.csv")

lap_times_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\lap_times.csv")
lap_times_df = lap_times_df[["raceId", "driverId", "lap", "milliseconds"]]

# calculate fastest lap time for each driver in each race
fastest_lap_times = lap_times_df.loc[lap_times_df.groupby(["raceId", "driverId"])['milliseconds'].idxmin()]

# convert milliseconds to seconds
fastest_lap_times['milliseconds'] = fastest_lap_times['milliseconds'] / 1000

# merge fastest lap times with combined_df
combined_df = pd.merge(combined_df, fastest_lap_times, on=["raceId", "driverId"], how="left")

# rename milliseconds to fLapTime
combined_df = combined_df.rename(columns={"milliseconds": "fLapTime"})

# save the combined dataset to a new csv file
combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_lap_times.csv", index=False)

del combined_df
del lap_times_df
del fastest_lap_times


In [38]:
# driver standings from driver_standings.csv for each race so we can see how many points the driver had after each round. Include drivertId, raceId, position, wins
driver_standings_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\archive\\driver_standings.csv")
driver_standings_df = driver_standings_df[["raceId", "driverId", "position", "wins", "points"]]

combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\combined_results_with_lap_times.csv")

# merge driver standings with combined_df
combined_df = pd.merge(combined_df, driver_standings_df, on=["raceId", "driverId"], how="left")

# rename points_y to accumPoints to indicate these are the points the driver had after each round and rename points_x to racePoints
combined_df = combined_df.rename(columns={"points_x": "racePoints"})
combined_df = combined_df.rename(columns={"points_y": "accumPoints"})

# these accumPoints also need to be normalized, add a column called nAccumPoints that normalizes them by the maximum points a driver had after each round (per race)
combined_df["nAccumPoints"] = combined_df.groupby("raceId")["accumPoints"].transform(lambda x: x / x.max() if x.max() else 0)

# drop resultId
combined_df = combined_df.drop(columns=["resultId"])

# save the combined dataset to a new csv file
combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data.csv", index=False)

# save one with only column names
combined_df.head(0).to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_25.csv", index=False)

del combined_df
del driver_standings_df

This leaves us with the final dataset for now, with the columns:
* worldChamp	
* driverId	
* driverRef
* resultId
* raceId
* constructorId
* startPos
* finishPos
* racePoints	
* year	
* round	
* nPoints	
* q1	
* q2	
* q3	
* nPits	# removed for redundancy
* pitDur # removed for redundancy
* avgLapTime	
* position	
* wins	
* accumPoints
* nAccumPoints

After some webscraper shenanigans, the final 2025 dataset is also done, but I need to give the races raceIds and I need to give them the normalized points

In [ ]:
# give 2025 dataset raceIds, nPoints, and nAccumPoints
f1_25 = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025.csv")

# add raceIds to the 16 races in 2025 so far. 2024 ended with 1144 so 2025 starts with 1145. Group by round and give each round a raceId
f1_25['raceId'] = f1_25.groupby("round").ngroup() + 1145
# add nPoints and nAccumPoints
f1_25["nPoints"] = f1_25.groupby("raceId")["racePoints"].transform(lambda x: x / x.max() if x.max() else 0)
f1_25["nAccumPoints"] = f1_25.groupby("raceId")["accumPoints"].transform(lambda x: x / x.max() if x.max() else 0)

# i also wanna add a column that indicates the difference between startPos and finishPos, call it posChange for both 2025 and the main dataset
f1_25["posChange"] = f1_25["startPos"] - f1_25["finishPos"]
# add posChange to the main dataset as well
combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data.csv")
combined_df["posChange"] = combined_df["startPos"] - combined_df["finishPos"]

# Change the order of the columns to have posChange after finishPos for both datasets worldChamp	driverId	driverRef	raceId	constructorId	startPos	finishPos	racePoints	year	round	circuitId	name	nPoints	q1	q2	q3	nPits	pitDur	lap	fLapTime	position	wins	accumPoints	nAccumPoints	posChange
cols = combined_df.columns.tolist()
finish_idx = cols.index('finishPos')
cols.insert(finish_idx + 1, 'posChange')
cols = [col for i, col in enumerate(cols) if col != 'posChange' or i == finish_idx + 1]
combined_df = combined_df[cols]
f1_25 = f1_25[cols]


combined_df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_datat.csv", index=False)
f1_25.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025t.csv", index=False)

del f1_25
del combined_df

That covers the data collection part of this project. Now we can move on to data cleanup

Since machine learning methods cannot function with missing values, and we have tons of missing values, we need to do something about that.

In [40]:
import pandas as pd

df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_datat.csv")
print(df.isnull().sum())

worldChamp           0
driverId             0
driverRef            0
raceId               0
constructorId        0
startPos             0
finishPos            0
posChange            0
racePoints           0
year                 0
round                0
circuitId            0
name                 0
nPoints              0
q1               16419
q2               20845
q3               22995
nPits            21172
pitDur           21212
lap              15710
fLapTime         15710
position           469
wins               469
accumPoints        469
nAccumPoints       469
dtype: int64


In [41]:
# I wanna make 2 versions of the dataset, one with the fastest lap time and qualifying times included and one without it because it has a lot of missing values
# I think pitstops has too many missing values to consider, so im complety dropping those columns. All data will now be in D:\Maastricht University\MSB1015\ML\Data
combined_df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_datat.csv")
df = combined_df.drop(columns=["nPits", "pitDur"])
df_no_flap = df.drop(columns=["fLapTime", "lap", "q1", "q2", "q3"])

df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_full.csv", index=False)
df_no_flap.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_no_times.csv", index=False)

del df
del combined_df
del df_no_flap

Lets go down the list of each category of missing values. I want to fix one first, the position, wins and accumpoints missing values. These occur at the beginning of new seasons when it doesnt know how to rank the drivers below the top10 that havent gotten points yet. For each raceId, I want to check if theres a value in all rows of column "position". If not, give it one below the highest value in the raceId.

In [42]:
# Fix position, wins, accumPoints, nAccumPoints missing values in f1_data_no_times and f1_data_full
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_no_times.csv")
df_full = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_full.csv")

def fill_missing_positions(group):
    # Find the max position in the group (race)
    max_pos = group['position'].max()
    # Assign missing positions to max_pos + 1
    group['position'] = group['position'].fillna(max_pos + 1)
    return group

df = df.groupby('raceId').apply(fill_missing_positions)
df_full = df_full.groupby('raceId').apply(fill_missing_positions)

# For wins, accumPoints, nAccumPoints: fill missing with 0 (since no points/wins yet)
df['wins'] = df['wins'].fillna(0)
df['accumPoints'] = df['accumPoints'].fillna(0)
df['nAccumPoints'] = df['nAccumPoints'].fillna(0)

df_full['wins'] = df_full['wins'].fillna(0)
df_full['accumPoints'] = df_full['accumPoints'].fillna(0)
df_full['nAccumPoints'] = df_full['nAccumPoints'].fillna(0)

# Save the cleaned dataset
df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_nt.csv", index=False)
df_full.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_full.csv", index=False)

del df
del df_full

C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20992\19903763.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('raceId').apply(fill_missing_positions)
C:\Users\Gebruiker\AppData\Local\Temp\ipykernel_20992\19903763.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_full = df_full.groupby('raceId').apply(fill_missing_positions)


In [43]:
# Recheck for missing values
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_full.csv")
print(df.isnull().sum())


worldChamp           0
driverId             0
driverRef            0
raceId               0
constructorId        0
startPos             0
finishPos            0
posChange            0
racePoints           0
year                 0
round                0
circuitId            0
name                 0
nPoints              0
q1               16419
q2               20845
q3               22995
lap              15710
fLapTime         15710
position             0
wins                 0
accumPoints          0
nAccumPoints         0
dtype: int64


In [45]:
# Recheck for missing values
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_nt.csv")
print(df.isnull().sum())

worldChamp       0
driverId         0
driverRef        0
raceId           0
constructorId    0
startPos         0
finishPos        0
posChange        0
racePoints       0
year             0
round            0
circuitId        0
name             0
nPoints          0
position         0
wins             0
accumPoints      0
nAccumPoints     0
dtype: int64


In [46]:
# Remake f1_25 with the same columns as f1_data_no_times_clean and f1_data_full_clean
f1_25 = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025t.csv")
f1_25['worldChamp'] = 0  # Set all to 0 for test data

f1_25_no_times = f1_25.drop(columns=["nPits", "pitDur", "fLapTime", "lap", "q1", "q2", "q3"])
f1_25_full = f1_25.drop(columns=["nPits", "pitDur"])


f1_25_no_times.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025_no_times.csv", index=False)
f1_25_full.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025_full.csv", index=False)


del f1_25
del f1_25_no_times
del f1_25_full

In [47]:
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025_no_times.csv")
print(df.isnull().sum())

# what rows are the missing values in finishPos and racePoints
output = df[df['finishPos'].isnull() | df['racePoints'].isnull()]
print(output)


worldChamp       0
driverId         0
driverRef        0
raceId           0
constructorId    0
startPos         0
finishPos        0
posChange        0
racePoints       0
year             0
round            0
circuitId        0
name             0
nPoints          0
position         0
wins             0
accumPoints      0
nAccumPoints     0
dtype: int64
Empty DataFrame
Columns: [worldChamp, driverId, driverRef, raceId, constructorId, startPos, finishPos, posChange, racePoints, year, round, circuitId, name, nPoints, position, wins, accumPoints, nAccumPoints]
Index: []


Now that all missing values have been removed, we can move on to encoding. In no_times, theres only one 2 columns which are categorical, but these can be removed anyways since driver name is stored in driverId and circuit name is stored in circuitId, so no encoding is required after these are dropped.

In [51]:
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_nt.csv")
df_25 = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025_no_times.csv")

df = df.drop(columns=["driverRef", "name"])
df_25 = df_25.drop(columns=["driverRef", "name"])

df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f1_nt.csv", index=False)
df_25.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f125_nt.csv", index=False)

If we instead of making for each year marking the world champion, we mark each driver if they have ever been champion, period, we should be able to predict if the 2025 drivers have a shot of ever becoming champion. In order to do this, we want to change worldChamp to be a 1 if the driver has ever been champ before

In [ ]:
import pandas as pd
import numpy as np

# Change worldChamp to be 1 if the driver has ever been a world champion in their career, not just that year
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\ML\\FinalData\\f1_nt.csv")
world_champions = pd.read_csv("d:\\Maastricht University\\MSB1015\\results\\world_champions.csv")
# Get a list of all unique world champion driverIds
champion_driver_ids = world_champions['driverId'].unique()
# Create a new column 'everWorldChamp' that is 1 if the driverId is in champion_driver_ids, else 0
df['everWorldChamp'] = df['driverId'].apply(lambda x: 1 if x in champion_driver_ids else 0)

# Replace worldChamp with everWorldChamp, and put it back in the first column
df = df.drop(columns=['worldChamp'])
df = df.rename(columns={'everWorldChamp': 'worldChamp'})
df = df[['worldChamp'] + [col for col in df.columns if col != 'worldChamp']]

df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\FinalData\\f1e_nt.csv", index=False)

Now we want to investigate the full data as well. Not just the points. Lets go through making sure we can use this data.

In [ ]:
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_full.csv")
print(df.isnull().sum())
del df

worldChamp           0
driverId             0
driverRef            0
raceId               0
constructorId        0
startPos             0
finishPos            0
posChange            0
racePoints           0
year                 0
round                0
circuitId            0
name                 0
nPoints              0
q1               16419
q2               20845
q3               22995
lap              15710
fLapTime         15710
position             0
wins                 0
accumPoints          0
nAccumPoints         0
dtype: int64


In [10]:
# Make the qualifying times into one column that indicates the average qualifying time
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1_data_full.csv") 

df['avgQualTime'] = df[['q1', 'q2', 'q3']].mean(axis=1)
df = df.drop(columns=['q1', 'q2', 'q3'])

# drop rows that dont have qualifying times
df = df.dropna(subset=['avgQualTime'])

df.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f1test_data_full.csv", index=False)

# apply to f12025_full as well
f1_25 = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025_full.csv")
f1_25['avgQualTime'] = f1_25[['q1', 'q2', 'q3']].mean(axis=1)
f1_25 = f1_25.drop(columns=['q1', 'q2', 'q3'])
f1_25 = f1_25.dropna(subset=['avgQualTime'])
print(f1_25.isnull().sum())

f1_25.to_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025test_full.csv", index=False)

worldChamp        0
driverId          0
driverRef         0
raceId            0
constructorId     0
startPos          0
finishPos         0
posChange         0
racePoints        0
year              0
round             0
circuitId         0
name              0
nPoints           0
lap              10
fLapTime         13
position          0
wins              0
accumPoints       0
nAccumPoints      0
avgQualTime       0
dtype: int64


As can be seen, we have some missing values here for flaptimes. We can encode for this for taking the average fLaptime per race. Also just remove the lap, we really dfont need it.

In [ ]:


# remove lap column, encode fLaptimes with the average fastest lap time per circuit
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f1test_data_full.csv")
df = df.drop(columns=['lap'])

# average fastest lap time per circuit
avg_fastest_lap_per_circuit = df.groupby('circuitId')['fLapTime'].mean().reset_index()

# for each row in df, if fLapTime is NaN, replace it with the average fastest lap time for that circuit
df = pd.merge(df, avg_fastest_lap_per_circuit, on='circuitId', suffixes=('', '_avg'))
df['fLapTime'] = df.apply(lambda row: row['fLapTime_avg'] if pd.isna(row['fLapTime']) else row['fLapTime'], axis=1)
df = df.drop(columns=['fLapTime_avg'])

# Drop rows with missing fLapTime values (mostly from 1994-1995 with poor data quality)
print(f"Shape before dropping missing fLapTime: {df.shape}")
df = df.dropna(subset=['fLapTime'])
print(f"Shape after dropping missing fLapTime: {df.shape}")

print("\nMissing values after dropping:")
print(df.isnull().sum())

df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f1_f.csv", index=False)



Shape before dropping missing fLapTime: (10080, 20)
Shape after dropping missing fLapTime: (9928, 20)

Missing values after dropping:
worldChamp       0
driverId         0
driverRef        0
raceId           0
constructorId    0
startPos         0
finishPos        0
posChange        0
racePoints       0
year             0
round            0
circuitId        0
name             0
nPoints          0
fLapTime         0
position         0
wins             0
accumPoints      0
nAccumPoints     0
avgQualTime      0
dtype: int64


In [12]:
# apply to f12025test_full as well
f1_25 = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\f12025test_full.csv")
f1_25 = f1_25.drop(columns=['lap'])
# for each row in f1_25, if fLapTime is NaN, replace it with the average fastest lap time for that circuit from df
f1_25 = pd.merge(f1_25, avg_fastest_lap_per_circuit, on='circuitId', suffixes=('', '_avg'))
f1_25['fLapTime'] = f1_25.apply(lambda row: row['fLapTime_avg'] if pd.isna(row['fLapTime']) else row['fLapTime'], axis=1)
f1_25 = f1_25.drop(columns=['fLapTime_avg'])

print(f1_25.isnull().sum())

f1_25.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f125_f.csv", index=False)

worldChamp       0
driverId         0
driverRef        0
raceId           0
constructorId    0
startPos         0
finishPos        0
posChange        0
racePoints       0
year             0
round            0
circuitId        0
name             0
nPoints          0
fLapTime         0
position         0
wins             0
accumPoints      0
nAccumPoints     0
avgQualTime      0
dtype: int64


Lastly, I want to create a version of the full datasets where instead of only one champion is appointed each season, if a driver ever was champion, they should be marked with one

In [1]:
import pandas as pd
import numpy as np

# make versions of f1_f and f125_f where a champion is marked with 1 if they have ever been one

df = pd.read_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f1_f.csv")
world_champions = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\world_champions.csv")
# Get a list of all unique world champion driverIds
champion_driver_ids = world_champions['driverId'].unique()
# Create a new column 'everWorldChamp' that is 1 if the driverId is in champion_driver_ids, else 0
df['everWorldChamp'] = df['driverId'].apply(lambda x: 1 if x in champion_driver_ids else 0)
# Replace worldChamp with everWorldChamp, and put it back in the first column
df = df.drop(columns=['worldChamp'])
df = df.rename(columns={'everWorldChamp': 'worldChamp'})
df = df[['worldChamp'] + [col for col in df.columns if col != 'worldChamp']]

df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f1e_f.csv", index=False)

# apply to f125_f as well
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f125_f.csv")
df['everWorldChamp'] = df['driverId'].apply(lambda x: 1 if x in champion_driver_ids else 0)
# Replace worldChamp with everWorldChamp, and put it back in the first column
df = df.drop(columns=['worldChamp'])
df = df.rename(columns={'everWorldChamp': 'worldChamp'})
df = df[['worldChamp'] + [col for col in df.columns if col != 'worldChamp']]

df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f125e_f.csv", index=False)

# Make a version of this as well on the no times dataset
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f1_nt.csv")
world_champions = pd.read_csv("d:\\Maastricht University\\MSB1015\\interval\\world_champions.csv")
# Get a list of all unique world champion driverIds
champion_driver_ids = world_champions['driverId'].unique()
# Create a new column 'everWorldChamp' that is 1 if the driverId is in champion_driver_ids, else 0
df['everWorldChamp'] = df['driverId'].apply(lambda x: 1 if x in champion_driver_ids else 0)
# Replace worldChamp with everWorldChamp, and put it back in the first column
df = df.drop(columns=['worldChamp'])
df = df.rename(columns={'everWorldChamp': 'worldChamp'})
df = df[['worldChamp'] + [col for col in df.columns if col != 'worldChamp']]
df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f1e_nt.csv", index=False)
# apply to f125_nt as well
df = pd.read_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f125_nt.csv")
df['everWorldChamp'] = df['driverId'].apply(lambda x: 1 if x in champion_driver_ids else 0)
# Replace worldChamp with everWorldChamp, and put it back in the first column
df = df.drop(columns=['worldChamp'])
df = df.rename(columns={'everWorldChamp': 'worldChamp'})
df = df[['worldChamp'] + [col for col in df.columns if col != 'worldChamp']]
df.to_csv("d:\\Maastricht University\\MSB1015\\ML\\Data\\f125e_nt.csv", index=False)
